# 01 - Ingesta Parquet a RAW

Objetivo del Punto 3: ingestar por mes (2015-2025, yellow/green) hacia esquema RAW en Snowflake con metadatos y auditoria.

In [1]:
import os
import requests
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T


In [2]:
spark_snowflake_pkg = os.getenv('SPARK_SNOWFLAKE_PACKAGE', 'net.snowflake:spark-snowflake_2.12:3.1.8')
snowflake_jdbc_pkg = os.getenv('SNOWFLAKE_JDBC_PACKAGE', 'net.snowflake:snowflake-jdbc:3.14.4')

spark = (
    SparkSession.builder
    .appName("PSet3 Ingesta RAW")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", f"{spark_snowflake_pkg},{snowflake_jdbc_pkg}")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

print(spark.version)
print('spark_snowflake_pkg=', spark_snowflake_pkg)
print('snowflake_jdbc_pkg=', snowflake_jdbc_pkg)

3.5.0
spark_snowflake_pkg= net.snowflake:spark-snowflake_2.12:3.1.8
snowflake_jdbc_pkg= net.snowflake:snowflake-jdbc:3.14.4


## 1) Configuracion por variables de entorno

In [3]:
def get_env(name, default=None, required=False):
    value = os.getenv(name, default)
    if required and (value is None or str(value).strip() == ""):
        raise ValueError(f"Falta variable de entorno requerida: {name}")
    return value

cfg = {
    "run_id": get_env("RUN_ID", "manual_local_001"),
    "start_year": int(get_env("START_YEAR", "2015")),
    "end_year": int(get_env("END_YEAR", "2025")),
    "months": [m.strip() for m in get_env("MONTHS", "01,02,03,04,05,06,07,08,09,10,11,12").split(",")],
    "services": [s.strip() for s in get_env("SERVICES", "yellow,green").split(",")],
    "parquet_base_url": get_env("PARQUET_BASE_URL", required=True),
    "taxi_zone_lookup_url": get_env("TAXI_ZONE_LOOKUP_URL", required=True),
    "validate_strict": get_env("VALIDATE_STRICT", "true").lower() == "true",
    "fail_on_missing_month": get_env("FAIL_ON_MISSING_MONTH", "false").lower() == "true",
}

cfg

{'run_id': 'manual_local_001',
 'start_year': 2015,
 'end_year': 2025,
 'months': ['01',
  '02',
  '03',
  '04',
  '05',
  '06',
  '07',
  '08',
  '09',
  '10',
  '11',
  '12'],
 'services': ['yellow', 'green'],
 'parquet_base_url': 'https://d37ci6vzurychx.cloudfront.net/trip-data',
 'taxi_zone_lookup_url': 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv',
 'validate_strict': True,
 'fail_on_missing_month': False}

In [4]:
snowflake_host = get_env("SNOWFLAKE_HOST", "").strip()
snowflake_account = get_env("SNOWFLAKE_ACCOUNT", "").strip()
snowflake_port = get_env("SNOWFLAKE_PORT", "443").strip()

if not snowflake_host:
    if not snowflake_account:
        raise ValueError("Debes configurar SNOWFLAKE_HOST o SNOWFLAKE_ACCOUNT")
    snowflake_host = f"{snowflake_account}.snowflakecomputing.com"

if snowflake_port and snowflake_port != "443":
    snowflake_host = f"{snowflake_host}:{snowflake_port}"

sf_options = {
    "sfURL": snowflake_host,
    "sfUser": get_env("SNOWFLAKE_USER", required=True),
    "sfPassword": get_env("SNOWFLAKE_PASSWORD", required=True),
    "sfRole": get_env("SNOWFLAKE_ROLE", required=True),
    "sfWarehouse": get_env("SNOWFLAKE_WAREHOUSE", required=True),
    "sfDatabase": get_env("SNOWFLAKE_DATABASE", required=True),
    "sfSchema": get_env("SNOWFLAKE_SCHEMA_RAW", "RAW"),
}

safe_preview = {k: ("***" if "Password" in k else v) for k, v in sf_options.items()}
safe_preview

{'sfURL': 'DVNRQOK-CRC15844.snowflakecomputing.com',
 'sfUser': 'tonyfajardo1d',
 'sfPassword': '***',
 'sfRole': 'ACCOUNTADMIN',
 'sfWarehouse': 'PSET3_WH',
 'sfDatabase': 'DM_PSET3',
 'sfSchema': 'RAW'}

In [5]:
pip install "pyarrow>=14.0.1"

Note: you may need to restart the kernel to use updated packages.


In [6]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "snowflake-connector-python"])
print("snowflake-connector-python instalado")

snowflake-connector-python instalado


In [7]:
import os, snowflake.connector
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
)
cur = conn.cursor()
cur.execute("select current_user(), current_account()")
print(cur.fetchone())
cur.close(); conn.close()

('TONYFAJARDO1D', 'TWC79991')


## 1.1) Bootstrap Snowflake (creacion automatica de DB/SCHEMAS)

In [8]:
sf_db = get_env("SNOWFLAKE_DATABASE", required=True)
sf_raw_schema = get_env("SNOWFLAKE_SCHEMA_RAW", "RAW")
sf_analytics_schema = get_env("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")
try:
    import snowflake.connector
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "snowflake-connector-python"])
    import snowflake.connector
account = get_env("SNOWFLAKE_ACCOUNT", "").strip()
host = get_env("SNOWFLAKE_HOST", "").strip()
conn_kwargs = {
    "user": get_env("SNOWFLAKE_USER", required=True),
    "password": get_env("SNOWFLAKE_PASSWORD", required=True),
    "role": get_env("SNOWFLAKE_ROLE", required=True),
    "warehouse": get_env("SNOWFLAKE_WAREHOUSE", required=True),
    "session_parameters": {"QUERY_TAG": f"PSET3_BOOTSTRAP_{cfg['run_id']}"},
}
# Solo agrega account/host si tienen valor real
if account:
    conn_kwargs["account"] = account
if host:
    conn_kwargs["host"] = host
if "account" not in conn_kwargs and "host" not in conn_kwargs:
    raise ValueError("Falta SNOWFLAKE_ACCOUNT o SNOWFLAKE_HOST")
conn = snowflake.connector.connect(**conn_kwargs)
bootstrap_sql = [
    f"CREATE DATABASE IF NOT EXISTS {sf_db}",
    f"CREATE SCHEMA IF NOT EXISTS {sf_db}.{sf_raw_schema}",
    f"CREATE SCHEMA IF NOT EXISTS {sf_db}.{sf_analytics_schema}",
]
with conn.cursor() as cur:
    for stmt in bootstrap_sql:
        cur.execute(stmt)
conn.close()
print(f"Bootstrap OK: {sf_db}, {sf_raw_schema}, {sf_analytics_schema}")

Bootstrap OK: DM_PSET3, RAW, ANALYTICS


In [9]:
ctx_q = """
SELECT
  CURRENT_ACCOUNT() AS account_name,
  CURRENT_USER() AS user_name,
  CURRENT_ROLE() AS role_name,
  CURRENT_WAREHOUSE() AS warehouse_name
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', ctx_q)
    .load()
    .show(truncate=False)
)

+------------+-------------+------------+--------------+
|ACCOUNT_NAME|USER_NAME    |ROLE_NAME   |WAREHOUSE_NAME|
+------------+-------------+------------+--------------+
|TWC79991    |TONYFAJARDO1D|ACCOUNTADMIN|PSET3_WH      |
+------------+-------------+------------+--------------+



## 2) Helpers de ingesta

In [10]:
base_dir = Path('/home/jovyan/work/data/raw_downloads')
base_dir.mkdir(parents=True, exist_ok=True)

def month_iter(start_year, end_year, months):
    for year in range(start_year, end_year + 1):
        for mm in months:
            yield year, mm

def build_trip_url(service_type, year, mm):
    file_name = f"{service_type}_tripdata_{year}-{mm}.parquet"
    return f"{cfg['parquet_base_url']}/{file_name}", file_name

def download_if_needed(url, local_path):
    if local_path.exists():
        return str(local_path), True

    r = requests.get(url, stream=True, timeout=120)
    if r.status_code == 404:
        return None, False
    r.raise_for_status()

    with open(local_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    return str(local_path), False


In [11]:
def standardize_schema(df, service_type):
    if service_type == 'yellow':
        pickup_col = 'tpep_pickup_datetime'
        dropoff_col = 'tpep_dropoff_datetime'
    elif service_type == 'green':
        pickup_col = 'lpep_pickup_datetime'
        dropoff_col = 'lpep_dropoff_datetime'
    else:
        raise ValueError(f'Servicio no soportado: {service_type}')

    cols = df.columns

    def c(name, cast_type=None):
        if name in cols:
            col = F.col(name)
            return col.cast(cast_type) if cast_type else col
        return F.lit(None).cast(cast_type) if cast_type else F.lit(None)

    out = (
        df.select(
            c(pickup_col, 'timestamp').alias('pickup_datetime'),
            c(dropoff_col, 'timestamp').alias('dropoff_datetime'),
            c('PULocationID', 'int').alias('pu_location_id'),
            c('DOLocationID', 'int').alias('do_location_id'),
            c('passenger_count', 'double').alias('passenger_count'),
            c('trip_distance', 'double').alias('trip_distance'),
            c('RatecodeID', 'int').alias('rate_code_id'),
            c('payment_type', 'int').alias('payment_type'),
            c('VendorID', 'int').alias('vendor_id'),
            c('trip_type', 'int').alias('trip_type'),
            c('store_and_fwd_flag', 'string').alias('store_and_fwd_flag'),
            c('fare_amount', 'double').alias('fare_amount'),
            c('extra', 'double').alias('extra'),
            c('mta_tax', 'double').alias('mta_tax'),
            c('tip_amount', 'double').alias('tip_amount'),
            c('tolls_amount', 'double').alias('tolls_amount'),
            c('improvement_surcharge', 'double').alias('improvement_surcharge'),
            c('congestion_surcharge', 'double').alias('congestion_surcharge'),
            c('Airport_fee', 'double').alias('airport_fee'),
            c('total_amount', 'double').alias('total_amount')
        )
        .withColumn('service_type', F.lit(service_type))
    )

    return out


In [12]:
def add_lineage_and_quality(df, source_year, source_month, source_path):
    now_utc = datetime.now(timezone.utc).isoformat()

    df2 = (
        df
        .withColumn('run_id', F.lit(cfg['run_id']))
        .withColumn('source_year', F.lit(int(source_year)))
        .withColumn('source_month', F.lit(str(source_month)))
        .withColumn('source_path', F.lit(source_path))
        .withColumn('ingested_at_utc', F.to_timestamp(F.lit(now_utc)))
        .withColumn(
            'trip_nk',
            F.sha2(
                F.concat_ws(
                    '||',
                    F.col('service_type'),
                    F.col('pickup_datetime').cast('string'),
                    F.col('dropoff_datetime').cast('string'),
                    F.col('pu_location_id').cast('string'),
                    F.col('do_location_id').cast('string'),
                    F.col('vendor_id').cast('string'),
                    F.col('total_amount').cast('string')
                ),
                256
            )
        )
    )

    if cfg['validate_strict']:
        df2 = (
            df2
            .filter(F.col('pickup_datetime').isNotNull())
            .filter(F.col('dropoff_datetime').isNotNull())
            .filter(F.col('pu_location_id').isNotNull())
            .filter(F.col('do_location_id').isNotNull())
            .filter(F.col('trip_distance').isNull() | (F.col('trip_distance') >= 0))
            .filter(F.col('total_amount').isNull() | (F.col('total_amount') >= 0))
            .filter(F.col('dropoff_datetime') >= F.col('pickup_datetime'))
        )

    return df2


## 3) Escritura a Snowflake RAW (idempotencia base)

In [13]:
raw_schema = get_env('SNOWFLAKE_SCHEMA_RAW', 'RAW')
raw_table = f"{raw_schema}.TRIPS"
audit_table = f"{raw_schema}.INGESTION_AUDIT"

def read_existing_trip_nk(service_type, source_year, source_month):
    query = f"""
        SELECT TRIP_NK
        FROM {raw_table}
        WHERE SERVICE_TYPE = '{service_type}'
          AND SOURCE_YEAR = {int(source_year)}
          AND SOURCE_MONTH = '{source_month}'
    """

    try:
        return (
            spark.read.format('snowflake')
            .options(**sf_options)
            .option('query', query)
            .load()
            .select(F.col('TRIP_NK').alias('trip_nk'))
        )
    except Exception:
        return spark.createDataFrame([], schema='trip_nk string')

def write_batch_to_raw(df_batch):
    (
        df_batch.write
        .format('snowflake')
        .options(**sf_options)
        .option('dbtable', raw_table)
        .mode('append')
        .save()
    )

def write_audit_row(rows):
    schema = T.StructType([
        T.StructField('run_id', T.StringType(), False),
        T.StructField('service_type', T.StringType(), False),
        T.StructField('source_year', T.IntegerType(), False),
        T.StructField('source_month', T.StringType(), False),
        T.StructField('source_path', T.StringType(), True),
        T.StructField('status', T.StringType(), False),
        T.StructField('records_in', T.LongType(), True),
        T.StructField('records_out', T.LongType(), True),
        T.StructField('error_message', T.StringType(), True),
        T.StructField('started_at_utc', T.TimestampType(), False),
        T.StructField('ended_at_utc', T.TimestampType(), False),
        T.StructField('duration_sec', T.DoubleType(), False),
    ])

    df_audit = spark.createDataFrame(rows, schema=schema)

    (
        df_audit.write
        .format('snowflake')
        .options(**sf_options)
        .option('dbtable', audit_table)
        .mode('append')
        .save()
    )


In [29]:
cfg["run_id"] = "idemp_test_2026_04_03"
cfg["services"] = ["yellow"]
cfg["start_year"] = 2024
cfg["end_year"] = 2024
cfg["months"] = ["12"]
print(cfg["run_id"], cfg["services"], cfg["start_year"], cfg["end_year"], cfg["months"])

idemp_test_2026_04_03 ['yellow'] 2024 2024 ['12']


## 4) Ejecucion de backfill mensual

In [14]:
audit_rows = []

for service_type in cfg['services']:
    for year, mm in month_iter(cfg['start_year'], cfg['end_year'], cfg['months']):
        started = datetime.now(timezone.utc)
        url, file_name = build_trip_url(service_type, year, mm)
        local_path = base_dir / file_name

        records_in = None
        records_out = None
        status = 'OK'
        error_message = None

        try:
            downloaded_path, from_cache = download_if_needed(url, local_path)

            if not downloaded_path:
                msg = f'No existe parquet para {service_type} {year}-{mm}'
                if cfg['fail_on_missing_month']:
                    raise FileNotFoundError(msg)
                status = 'MISSING_SOURCE'
                error_message = msg
            else:
                df_src = spark.read.parquet(downloaded_path)
                records_in = df_src.count()

                df_std = standardize_schema(df_src, service_type)
                df_lineage = add_lineage_and_quality(df_std, year, mm, url)
                df_lineage = df_lineage.dropDuplicates(['trip_nk'])

                existing_nk = read_existing_trip_nk(service_type, year, mm)
                df_new = df_lineage.join(existing_nk, on='trip_nk', how='left_anti')
                df_new = df_new.dropDuplicates(['trip_nk'])

                records_out = df_new.count()

                if records_out > 0:
                    write_batch_to_raw(df_new)

                if from_cache:
                    print(f'[CACHE] {service_type} {year}-{mm} in={records_in} out={records_out}')
                else:
                    print(f'[DOWNLOADED] {service_type} {year}-{mm} in={records_in} out={records_out}')

        except Exception as e:
            status = 'FAILED'
            error_message = str(e)[:4000]
            print(f'[ERROR] {service_type} {year}-{mm}: {error_message}')

        ended = datetime.now(timezone.utc)
        duration_sec = (ended - started).total_seconds()

        audit_rows.append((
            cfg['run_id'],
            service_type,
            int(year),
            mm,
            url,
            status,
            records_in,
            records_out,
            error_message,
            started,
            ended,
            float(duration_sec),
        ))

write_audit_row(audit_rows)
print(f'Auditoria escrita: {len(audit_rows)} lotes')

[CACHE] yellow 2015-01 in=12741035 out=0
[CACHE] yellow 2015-02 in=12442394 out=0
[CACHE] yellow 2015-03 in=13342951 out=0
[CACHE] yellow 2015-04 in=13063758 out=0
[CACHE] yellow 2015-05 in=13157677 out=0
[CACHE] yellow 2015-06 in=12324936 out=0
[CACHE] yellow 2015-07 in=11559666 out=0
[CACHE] yellow 2015-08 in=11123123 out=0
[CACHE] yellow 2015-09 in=11218122 out=0
[CACHE] yellow 2015-10 in=12307333 out=0
[CACHE] yellow 2015-11 in=11305240 out=0
[CACHE] yellow 2015-12 in=11452996 out=0
[CACHE] yellow 2016-01 in=10905067 out=0
[CACHE] yellow 2016-02 in=11375412 out=0
[CACHE] yellow 2016-03 in=12203824 out=0
[CACHE] yellow 2016-04 in=11927996 out=0
[CACHE] yellow 2016-05 in=11832049 out=0
[CACHE] yellow 2016-06 in=11131645 out=0
[CACHE] yellow 2016-07 in=10294080 out=0
[CACHE] yellow 2016-08 in=9942263 out=0
[CACHE] yellow 2016-09 in=10116018 out=0
[CACHE] yellow 2016-10 in=10854626 out=0
[CACHE] yellow 2016-11 in=10102128 out=0
[CACHE] yellow 2016-12 in=10446697 out=0
[CACHE] yellow 20

In [31]:
# Reingesta puntual: solo yellow 2024-12
cfg["run_id"] = "idemp_test_2026_04_03"
cfg["services"] = ["yellow"]
cfg["start_year"] = 2024
cfg["end_year"] = 2024
cfg["months"] = ["12"]
print("RUN:", cfg["run_id"])
print("TARGET:", cfg["services"], cfg["start_year"], cfg["months"])
audit_rows = []
for service_type in cfg["services"]:
    for year, mm in month_iter(cfg["start_year"], cfg["end_year"], cfg["months"]):
        started = datetime.now(timezone.utc)
        url, file_name = build_trip_url(service_type, year, mm)
        local_path = base_dir / file_name
        records_in = None
        records_out = None
        status = "OK"
        error_message = None
        try:
            downloaded_path, from_cache = download_if_needed(url, local_path)
            if not downloaded_path:
                msg = f"No existe parquet para {service_type} {year}-{mm}"
                if cfg["fail_on_missing_month"]:
                    raise FileNotFoundError(msg)
                status = "MISSING_SOURCE"
                error_message = msg
            else:
                df_src = spark.read.parquet(downloaded_path)
                records_in = df_src.count()
                df_std = standardize_schema(df_src, service_type)
                df_lineage = add_lineage_and_quality(df_std, year, mm, url).dropDuplicates(["trip_nk"])
                existing_nk = read_existing_trip_nk(service_type, year, mm)
                df_new = df_lineage.join(existing_nk, on="trip_nk", how="left_anti").dropDuplicates(["trip_nk"])
                records_out = df_new.count()
                if records_out > 0:
                    write_batch_to_raw(df_new)
                tag = "CACHE" if from_cache else "DOWNLOADED"
                print(f"[{tag}] {service_type} {year}-{mm} in={records_in} out={records_out}")
        except Exception as e:
            status = "FAILED"
            error_message = str(e)[:4000]
            print(f"[ERROR] {service_type} {year}-{mm}: {error_message}")
        ended = datetime.now(timezone.utc)
        duration_sec = (ended - started).total_seconds()
        audit_rows.append((
            cfg["run_id"],
            service_type,
            int(year),
            mm,
            url,
            status,
            records_in,
            records_out,
            error_message,
            started,
            ended,
            float(duration_sec),
        ))
write_audit_row(audit_rows)
print(f"Auditoria escrita: {len(audit_rows)} lote(s)")

RUN: idemp_test_2026_04_03
TARGET: ['yellow'] 2024 ['12']
[CACHE] yellow 2024-12 in=3668371 out=3597775
Auditoria escrita: 1 lote(s)


In [32]:
q_check_part = """
SELECT COUNT(*) AS c
FROM RAW.TRIPS
WHERE service_type='yellow'
  AND source_year=2024
  AND source_month='12'
"""
spark.read.format("snowflake").options(**sf_options).option("query", q_check_part).load().show()

+-------+
|      C|
+-------+
|3597775|
+-------+



In [33]:
q_audit_run = f"""
SELECT run_id, service_type, source_year, source_month, status, records_in, records_out, error_message
FROM RAW.INGESTION_AUDIT
WHERE run_id = '{cfg["run_id"]}'
ORDER BY started_at_utc DESC
"""
spark.read.format("snowflake").options(**sf_options).option("query", q_audit_run).load().show(truncate=False)

+---------------------+------------+-----------+------------+------+----------+-----------+-------------+
|RUN_ID               |SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|STATUS|RECORDS_IN|RECORDS_OUT|ERROR_MESSAGE|
+---------------------+------------+-----------+------------+------+----------+-----------+-------------+
|idemp_test_2026_04_03|yellow      |2024       |12          |OK    |3668371   |3597775    |NULL         |
+---------------------+------------+-----------+------------+------+----------+-----------+-------------+



## 5) Verificaciones rapidas de Punto 3

In [15]:
q_raw = f"""
SELECT SERVICE_TYPE, SOURCE_YEAR, SOURCE_MONTH, COUNT(*) AS ROWS_RAW
FROM {raw_table}
GROUP BY 1,2,3
ORDER BY 1,2,3
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_raw)
    .load()
    .show(30, truncate=False)
)

+------------+-----------+------------+--------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|ROWS_RAW|
+------------+-----------+------------+--------+
|green       |2015       |01          |1506268 |
|green       |2015       |02          |1572532 |
|green       |2015       |03          |1720117 |
|green       |2015       |04          |1661968 |
|green       |2015       |05          |1784221 |
|green       |2015       |06          |1636378 |
|green       |2015       |07          |1539148 |
|green       |2015       |08          |1529833 |
|green       |2015       |09          |1492510 |
|green       |2015       |10          |1627603 |
|green       |2015       |11          |1526922 |
|green       |2015       |12          |1605152 |
|green       |2016       |01          |1442315 |
|green       |2016       |02          |1507812 |
|green       |2016       |03          |1573184 |
|green       |2016       |04          |1540967 |
|green       |2016       |05          |1533961 |
|green       |2016  

In [16]:
q_audit = f"""
SELECT STATUS, COUNT(*) AS LOTES
FROM {audit_table}
WHERE RUN_ID = '{cfg['run_id']}'
GROUP BY 1
ORDER BY 1
"""

(
    spark.read.format('snowflake')
    .options(**sf_options)
    .option('query', q_audit)
    .load()
    .show(truncate=False)
)

+------+-----+
|STATUS|LOTES|
+------+-----+
|OK    |264  |
+------+-----+



In [17]:
from pathlib import Path
p = Path("/home/jovyan/work/data/raw_downloads")
print(p.exists(), len(list(p.glob("*.parquet"))))

True 264


In [19]:
# Test 2: verificar que exista data cargada en RAW.TRIPS
q = f"""
SELECT COUNT(*) AS total_rows
FROM {raw_table}
"""
(
    spark.read.format("snowflake")
    .options(**sf_options)
    .option("query", q)
    .load()
    .show()
)
# Test 3: auditoría de tu run
q_audit = f"""
SELECT RUN_ID, STATUS, COUNT(*) AS lotes
FROM {audit_table}
WHERE RUN_ID = '{cfg["run_id"]}'
GROUP BY 1,2
ORDER BY 2
"""
(
    spark.read.format("snowflake")
    .options(**sf_options)
    .option("query", q_audit)
    .load()
    .show(truncate=False)
)

+----------+
|TOTAL_ROWS|
+----------+
| 866674985|
+----------+

+----------------+------+-----+
|RUN_ID          |STATUS|LOTES|
+----------------+------+-----+
|manual_local_001|OK    |264  |
+----------------+------+-----+



In [23]:
# Mini-run solo para reingesta puntual
cfg["services"] = ["yellow"]
cfg["start_year"] = 2024
cfg["end_year"] = 2024
cfg["months"] = ["12"]
print(cfg["services"], cfg["start_year"], cfg["end_year"], cfg["months"])

['yellow'] 2024 2024 ['12']
